In [ ]:
# CELL 45-check — list what is attached under /kaggle/input and find the baseline files anywhere
import os, glob
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")
print("\npdfrate_features.csv anywhere:", glob.glob("/kaggle/input/**/pdfrate_features.csv", recursive=True))
print("hidost_cache.tar.gz anywhere:", glob.glob("/kaggle/input/**/hidost_cache.tar.gz", recursive=True))

In [ ]:
# CELL 45 — Baseline evaluation. PDFrate (original Mimicus features; Random Forest as in Mimicus/PDFrate: 1000 trees, 43 features per split) and Hidost (authors' cached structural paths; train-only selection, paths in >= 1000 files as in the paper; Random Forest 200 trees as in the paper) vs our LightGBM (exact CELL 23 setup). All scored with CELL 23's metric functions on the same dev files.
import os, glob, tarfile, warnings
from collections import Counter
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

OUT_DIR = "/kaggle/working/baseline_results"
os.makedirs(OUT_DIR, exist_ok=True)

def find_one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert len(hits) == 1, f"{what}: expected exactly 1 file for {pattern}, found {hits} - check the attached inputs"
    return hits[0]

FF = find_one("/kaggle/input/**/full/full_features.csv", "full_features.csv (BTP Full)")
PRF = find_one("/kaggle/input/**/big/pdfrate_features.csv", "pdfrate_features.csv (BTP baselines)")
STF = find_one("/kaggle/input/**/big/baseline_status.csv", "baseline_status.csv (BTP baselines)")
tars = sorted(glob.glob("/kaggle/input/**/big/hidost_cache.tar.gz", recursive=True))
dirs = sorted(glob.glob("/kaggle/input/**/big/hidost_cache", recursive=True))
assert tars or dirs, "Hidost cache not found (hidost_cache.tar.gz or an extracted hidost_cache folder) - check BTP baselines input"
print("Inputs:\n ", FF, "\n ", PRF, "\n ", STF, "\n ", tars[0] if tars else dirs[0])

# ---------- our features + CELL 23 feature groups (copied exactly) ----------
df = pd.read_csv(FF)
META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID",
      "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
FULL = sorted(set(META + L1 + L23 + L4 + L5))
assert not [c for c in FULL if c not in df.columns], "feature columns missing in full_features.csv"
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
PAIRS = [("G1_resave", "F1_digit_resave"), ("G2_recompress", "F2_digit_recompress"), ("G3_note_incr", "F3_digit_incr"),
         ("G1_resave", "F4_value_resave"), ("G1_resave", "F5_cover_replace"), ("G4_linearize", "F6_digit_linearize"),
         ("G5_stamp_honest", "F7_stamp_fake"), ("G6_textbox_honest", "F8_textbox_fake")]

def tpr_at(y, s, fpr):                               # CELL 23
    thr = np.quantile(s[y == 0], 1 - fpr)
    return float((s[y == 1] > thr).mean())

def auc_or_nan(y, s):                                # CELL 23
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

def scores(y, s):                                    # CELL 23
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    return {"ROC-AUC": auc_or_nan(y, s), "PR-AUC": round(average_precision_score(y, s), 3),
            "Det@1%FA": round(tpr_at(y, s, 0.01), 3), "Det@5%FA": round(tpr_at(y, s, 0.05), 3)}

# ---------- PDFrate features (original Mimicus, 135) ----------
pr = pd.read_csv(PRF)
REFCOLS = ["out_file", "orig_file", "family", "variant", "label", "split", "twin", "edit_page"]
PR_FEATS = [c for c in pr.columns if c not in REFCOLS]
assert len(PR_FEATS) == 135, f"expected 135 PDFrate features, found {len(PR_FEATS)}"
n_nan = int(pr[PR_FEATS].isna().sum().sum())
pr[PR_FEATS] = pr[PR_FEATS].fillna(-1.0)
pr = pr[["out_file"] + PR_FEATS].rename(columns={c: "pdfrate__" + c for c in PR_FEATS})
PR_COLS = ["pdfrate__" + c for c in PR_FEATS]
print(f"\nPDFrate: {len(pr)} files x {len(PR_FEATS)} features | missing values filled with -1: {n_nan}")

# ---------- Hidost caches (authors' pdf2vals output, cached by authors' cacher) ----------
st = pd.read_csv(STF)
h_ok = set(st.loc[st.hidost_ok == True, "out_file"])

def parse_cache(data):                               # '<path ending \0\0> <value>\n' -> {path: value}
    out, i = {}, 0
    while True:
        j = data.find(b"\0\0 ", i)
        if j < 0:
            break
        k = data.find(b"\n", j)
        k = len(data) if k < 0 else k
        out[data[i:j + 2]] = float(data[j + 3:k])
        i = k + 1
    return out

caches = {}
if tars:
    with tarfile.open(tars[0], "r:gz") as tf:
        for m in tqdm(tf, desc="Reading Hidost cache"):
            b = os.path.basename(m.name)
            if m.isfile() and b in h_ok:
                caches[b] = parse_cache(tf.extractfile(m).read())
else:
    for root, _, files in os.walk(dirs[0]):
        for fn in files:
            if fn in h_ok:
                caches[fn] = parse_cache(open(os.path.join(root, fn), "rb").read())
assert len(caches) == len(h_ok), f"Hidost cache files read: {len(caches)}, expected {len(h_ok)}"
print(f"Hidost: cache read for {len(caches)} files")

# ---------- one table, one row per file ----------
data = df.merge(pr, on="out_file", how="left")
data["has_pdfrate"] = data[PR_COLS[0]].notna()
data["has_hidost"] = data.out_file.isin(caches.keys())
tr_all = data[data.split == "train"].reset_index(drop=True)
te_all = data[data.split == "dev"].reset_index(drop=True)
COMMON = te_all.has_pdfrate & te_all.has_hidost
print(f"\nCoverage (dev files): all {len(te_all)} | PDFrate {int(te_all.has_pdfrate.sum())} | Hidost {int(te_all.has_hidost.sum())}"
      f" | both (used for the comparison) {int(COMMON.sum())}")

# ---------- Hidost: train-only feature selection (paper: paths in >= 1000 training files) + vectorization as feat-extract ----------
tr_h = tr_all[tr_all.has_hidost].out_file.tolist()
cnt = Counter()
for f in tr_h:
    cnt.update(caches[f].keys())
M_LIST = {"Hidost (paper: paths in >= 1000 train files)": 1000,
          f"Hidost (paths in >= 1% of train files = {round(0.01 * len(tr_h))})": round(0.01 * len(tr_h)),
          "Hidost (paths in >= 100 train files)": 100}

def vectorize(files, feats):                         # feat-extract --values: 6 significant digits, 0 if path absent
    idx = {p: i for i, p in enumerate(feats)}
    X = np.zeros((len(files), len(feats)))
    for r, f in enumerate(files):
        for p, v in caches[f].items():
            j = idx.get(p)
            if j is not None:
                X[r, j] = float("%g" % v)
    return X

HID = {}
for name, m in M_LIST.items():
    feats = sorted(p for p, n in cnt.items() if n >= m)
    HID[name] = feats
    print(f"{name}: {len(feats)} structural paths kept (of {len(cnt)} seen in train)")
with open(f"{OUT_DIR}/hidost_paths_selected_m1000.txt", "w") as fh:
    fh.write("\n".join(p.rstrip(b"\0").replace(b"\0", b"/").decode("latin-1") for p in HID[list(M_LIST)[0]]))

# ---------- train all methods on train documents, score dev ----------
SC = {}                                              # method -> dev scores (aligned with te_all; NaN where no features)

def put(name, mask, s):
    v = np.full(len(te_all), np.nan)
    v[mask.values] = s
    SC[name] = v

allm = pd.Series(True, index=te_all.index)
put("Rule: %%EOF count > 1 (industry check)", allm, te_all.n_eof.values)
put("Rule: true revisions > 1", allm, te_all.true_revisions.values)
print("\nTraining ...")
m_l1 = lgb.LGBMClassifier(**PARAMS).fit(tr_all[L1], tr_all.label)
put("Coding style (L1 file-style features, LightGBM)", allm, m_l1.predict_proba(te_all[L1])[:, 1])
m_full = lgb.LGBMClassifier(**PARAMS).fit(tr_all[FULL], tr_all.label)
put("OURS: LightGBM FULL MODEL (L1-L5)", allm, m_full.predict_proba(te_all[FULL])[:, 1])

trp, tep = tr_all[tr_all.has_pdfrate], te_all.has_pdfrate
rf_pr = RandomForestClassifier(n_estimators=1000, max_features=43, random_state=42, n_jobs=-1)   # Mimicus RandomForest.py ("Used by PDFrate")
rf_pr.fit(trp[PR_COLS].values, trp.label.values)
put("PDFrate (Mimicus reimplementation, RF 1000 trees)", tep, rf_pr.predict_proba(te_all.loc[tep, PR_COLS].values)[:, 1])

trh, teh = tr_all[tr_all.has_hidost], te_all.has_hidost
for name, feats in tqdm(HID.items(), desc="Hidost models"):
    if not feats:
        print(f"{name}: no features - skipped")
        continue
    rf_h = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)                     # Hidost paper: 200 trees, other sklearn defaults
    rf_h.fit(vectorize(trh.out_file.tolist(), feats), trh.label.values)
    put(name, teh, rf_h.predict_proba(vectorize(te_all.loc[teh, "out_file"].tolist(), feats))[:, 1])

# ---------- sanity: our model on ALL dev files must reproduce the reported numbers ----------
EV_ALL = ~te_all.variant.isin(NO_EV)
o = SC["OURS: LightGBM FULL MODEL (L1-L5)"]
a_all, a_ev = auc_or_nan(te_all.label, o), auc_or_nan(te_all.label[EV_ALL], o[EV_ALL.values])
print(f"\nSanity - our model on all {len(te_all)} dev files: AUC all versions {a_all} (reported 0.908) | "
      f"evidence-bearing {a_ev} (reported 0.998) -> {'✅ matches' if abs(a_all - 0.908) <= 0.002 and abs(a_ev - 0.998) <= 0.002 else '⚠️ differs - tell Claude'}")

# ---------- results on the common dev files ----------
te = te_all[COMMON].reset_index(drop=True)
S = {k: v[COMMON.values] for k, v in SC.items()}
EVID = ~te.variant.isin(NO_EV)
ORDER = list(SC.keys())

t_all = pd.DataFrame([{"Method": k, **scores(te.label, S[k])} for k in ORDER])
t_ev = pd.DataFrame([{"Method": k, **scores(te.label[EVID], S[k][EVID.values])} for k in ORDER])
twin = []
for k in ORDER:
    row = {"Method": k}
    for g, f in PAIRS:
        msk = te.variant.isin([g, f]).values
        row[f.split("_")[0] + " vs " + g.split("_")[0]] = auc_or_nan(te.label[msk], S[k][msk])
    twin.append(row)
t_twin = pd.DataFrame(twin)

rng = np.random.default_rng(0)
docs = te.orig_file.unique()
by_doc = {d: te.index[te.orig_file == d].values for d in docs}
boots = {k: ([], []) for k in ORDER}
for _ in tqdm(range(300), desc="Bootstrap CIs"):
    idx = np.concatenate([by_doc[d] for d in rng.choice(docs, size=len(docs), replace=True)])
    y, ev = te.label.values[idx], EVID.values[idx]
    for k in ORDER:
        s = S[k][idx]
        boots[k][0].append(roc_auc_score(y, s))
        boots[k][1].append(roc_auc_score(y[ev], s[ev]))
ci = pd.DataFrame([{"Method": k,
                    "AUC all [95% CI]": f"{np.mean(a):.3f} [{np.percentile(a, 2.5):.3f}, {np.percentile(a, 97.5):.3f}]",
                    "AUC evidence-bearing [95% CI]": f"{np.mean(e):.3f} [{np.percentile(e, 2.5):.3f}, {np.percentile(e, 97.5):.3f}]"}
                   for k, (a, e) in boots.items()])

pd.set_option("display.width", 250)
print(f"\n=== B1: ALL 15 VERSIONS (dev files with every method's features: {len(te)} files, {te.orig_file.nunique()} documents) ===")
print(t_all.to_string(index=False))
print("\n=== B2: EVIDENCE-BEARING ONLY (excl. F2/F7/F8 and twins G2/G5/G6) ===")
print(t_ev.to_string(index=False))
print("\n=== B3: TWIN AUC - forgery vs its honest twin (0.5 = cannot tell them apart) ===")
print(t_twin.to_string(index=False))
print("\n=== B4: 95% CONFIDENCE INTERVALS (bootstrap over dev documents, 300 resamples) ===")
print(ci.to_string(index=False))

t_all.to_csv(f"{OUT_DIR}/B1_all_versions.csv", index=False)
t_ev.to_csv(f"{OUT_DIR}/B2_evidence_bearing.csv", index=False)
t_twin.to_csv(f"{OUT_DIR}/B3_twin_auc.csv", index=False)
ci.to_csv(f"{OUT_DIR}/B4_confidence_intervals.csv", index=False)
pd.concat([te_all[["out_file", "orig_file", "family", "variant", "label", "tool"]], pd.DataFrame(SC)], axis=1) \
  .to_csv(f"{OUT_DIR}/dev_scores_all_methods.csv", index=False)
with open(f"{OUT_DIR}/BASELINE_RESULTS.txt", "w") as fh:
    fh.write(f"Coverage (dev files): all {len(te_all)} | PDFrate {int(te_all.has_pdfrate.sum())} | Hidost {int(te_all.has_hidost.sum())} | both {int(COMMON.sum())}\n")
    fh.write(f"Sanity - ours on all dev files: AUC all {a_all} | evidence-bearing {a_ev}\n\n")
    for title, t in [("B1 ALL VERSIONS", t_all), ("B2 EVIDENCE-BEARING", t_ev), ("B3 TWIN AUC", t_twin), ("B4 CONFIDENCE INTERVALS", ci)]:
        fh.write(f"=== {title} ===\n{t.to_string(index=False)}\n\n")
print(f"\nSaved in {OUT_DIR}: B1-B4 CSVs, dev_scores_all_methods.csv, hidost_paths_selected_m1000.txt, BASELINE_RESULTS.txt")

In [ ]:
# CELL 48 — FINAL baseline table: add TruFor to the CELL 45 methods and recompute every metric on the SAME dev files for all methods (files that every method scored; documents with all 15 versions). Uses CELL 45's saved per-file scores and CELL 23 metric functions.
import os, glob
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

assert all(k in globals() for k in ("scores", "auc_or_nan", "NO_EV", "PAIRS", "OUT_DIR")), "Run CELL 45 first (same session)"
DS = f"{OUT_DIR}/dev_scores_all_methods.csv"
assert os.path.exists(DS), "dev_scores_all_methods.csv missing - run CELL 45 first"
hits = sorted(glob.glob("/kaggle/input/**/trufor/full/trufor_dev_scores.csv", recursive=True))
assert len(hits) == 1, f"trufor_dev_scores.csv: expected 1, found {hits} - attach 'BTP baseline trufor' as input"

dev = pd.read_csv(DS)
tf = pd.read_csv(hits[0], usecols=["out_file", "trufor_score"])
METAC = ["out_file", "orig_file", "family", "variant", "label", "tool"]
METHODS = [c for c in dev.columns if c not in METAC]
TRUFOR = "TruFor (image forensics, CVPR 2023; official weights)"
dev = dev.merge(tf.rename(columns={"trufor_score": TRUFOR}), on="out_file", how="left")
METHODS.append(TRUFOR)

ok = dev[METHODS].notna().all(axis=1)
n15 = dev[ok].groupby("orig_file").out_file.count()
keep_docs = set(n15[n15 == 15].index)
te = dev[ok & dev.orig_file.isin(keep_docs)].reset_index(drop=True)
print(f"Dev files: {len(dev)} | scored by every method: {int(ok.sum())} | used (documents with all 15 versions): "
      f"{len(te)} files, {te.orig_file.nunique()} documents")
for m in METHODS:
    print(f"   coverage {m:58s} {int(dev[m].notna().sum())} files")

EVID = ~te.variant.isin(NO_EV)
t_all = pd.DataFrame([{"Method": m, **scores(te.label, te[m].values)} for m in METHODS])
t_ev = pd.DataFrame([{"Method": m, **scores(te.label[EVID], te[m].values[EVID.values])} for m in METHODS])
rows = []
for m in METHODS:
    r = {"Method": m}
    for g, f in PAIRS:
        msk = te.variant.isin([g, f]).values
        r[f.split("_")[0] + " vs " + g.split("_")[0]] = auc_or_nan(te.label[msk], te[m].values[msk])
    rows.append(r)
t_twin = pd.DataFrame(rows)

rng = np.random.default_rng(0)
docs = te.orig_file.unique()
by_doc = {d: te.index[te.orig_file == d].values for d in docs}
B = {m: ([], []) for m in METHODS}
for _ in range(300):
    idx = np.concatenate([by_doc[d] for d in rng.choice(docs, size=len(docs), replace=True)])
    y, ev = te.label.values[idx], EVID.values[idx]
    for m in METHODS:
        s = te[m].values[idx]
        B[m][0].append(roc_auc_score(y, s))
        B[m][1].append(roc_auc_score(y[ev], s[ev]))
ci = pd.DataFrame([{"Method": m,
                    "AUC all [95% CI]": f"{np.mean(a):.3f} [{np.percentile(a, 2.5):.3f}, {np.percentile(a, 97.5):.3f}]",
                    "AUC evidence-bearing [95% CI]": f"{np.mean(e):.3f} [{np.percentile(e, 2.5):.3f}, {np.percentile(e, 97.5):.3f}]"}
                   for m, (a, e) in B.items()])

summary = pd.DataFrame({"Method": METHODS,
                        "AUC all": t_all["ROC-AUC"].values,
                        "AUC evidence": t_ev["ROC-AUC"].values,
                        "Det@1%FA evidence": t_ev["Det@1%FA"].values,
                        "Twin F1": t_twin["F1 vs G1"].values,
                        "Twin F4": t_twin["F4 vs G1"].values,
                        "Twin F6": t_twin["F6 vs G4"].values})

pd.set_option("display.width", 250)
out = [f"Same dev files for every method: {len(te)} files, {te.orig_file.nunique()} documents\n"]
for title, t in [("FINAL SUMMARY (paper table)", summary), ("F1: ALL 15 VERSIONS", t_all),
                 ("F2: EVIDENCE-BEARING ONLY (excl. F2/F7/F8 and twins G2/G5/G6)", t_ev),
                 ("F3: TWIN AUC - forgery vs its honest twin (0.5 = cannot tell them apart)", t_twin),
                 ("F4: 95% CONFIDENCE INTERVALS (bootstrap over dev documents, 300 resamples)", ci)]:
    s = t.to_string(index=False)
    print(f"\n=== {title} ===\n{s}")
    out.append(f"=== {title} ===\n{s}\n")

summary.to_csv(f"{OUT_DIR}/FINAL_summary.csv", index=False)
t_all.to_csv(f"{OUT_DIR}/FINAL_F1_all_versions.csv", index=False)
t_ev.to_csv(f"{OUT_DIR}/FINAL_F2_evidence_bearing.csv", index=False)
t_twin.to_csv(f"{OUT_DIR}/FINAL_F3_twin_auc.csv", index=False)
ci.to_csv(f"{OUT_DIR}/FINAL_F4_confidence_intervals.csv", index=False)
te.to_csv(f"{OUT_DIR}/FINAL_dev_scores_common.csv", index=False)
with open(f"{OUT_DIR}/FINAL_BASELINE_TABLE.txt", "w") as fh:
    fh.write("\n".join(out))
print(f"\nSaved in {OUT_DIR}: FINAL_summary.csv, FINAL_F1-F4 CSVs, FINAL_dev_scores_common.csv, FINAL_BASELINE_TABLE.txt")